# Grid-Aware MADRL Training Notebook（配电网潮流约束版）

SimBench `1-LV-rural1--0-sw` 拓扑，3 个 prosumer agent，每步运行一次
Newton-Raphson 潮流，奖励含电压 / 线路约束惩罚。

## 与 train_madrl.ipynb 的主要差异

| 项目 | train_madrl | **train_madrl_grid** |
|------|-------------|----------------------|
| 环境 | `energy_storage` | **`grid_pf`**（调用 pandapower）|
| 奖励 | `composite`（5 分量）| **`grid_composite`**（+r_v_pen / r_l_pen）|
| num_envs | 12（多进程）| **1**（pandapower 暂不支持多进程）|
| 评估 | `evaluate_runner` | **自定义循环**（episode_recorder + grid_recorder）|
| 可视化 | 奖励分解 + SoC 轨迹 | + **电压时序图 / 复合图 / 违规统计图** |

## ⚠ 性能说明

- `num_envs=1` + pandapower ≈50 ms/step → 训练速度约为普通版的 **1/12**
- `train_episodes=100` 约需 3–5 分钟
- simbench 网络首次加载需 3–5 秒（有模块级缓存，重启 kernel 后只慢一次）

In [ ]:
from pathlib import Path
import sys

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root

In [ ]:
import numpy as np

from scripts.utils.experiment_notebook_utils import (
    build_runner,
    get_madrl_checkpoint_root,
    summarize_cfg,
)
from scripts.utils.torch_runtime import configure_torch_runtime, describe_device
from configs import compose_experiment_config
from configs.profiles import apply_grid_profile
from controllers import MADRLController
from scripts.recorders.episode_recorder import append_step_record, init_episode_record
from scripts.plots.grid_plots import (
    plot_battery_and_grid,
    plot_grid_constraint_summary,
    plot_node_voltages,
)
from scripts.recorders.grid_recorder import append_grid_step_record, init_grid_record
from scripts.plots.plots import plot_last_k_episodes_price_action_soc
from scripts.plots.reward_plots import plot_reward_decomposition

In [ ]:
# ── 训练规模 ──────────────────────────────────────────────────
algorithm          = "MADDPG"   # MADDPG | MATD3
train_episodes     = 100        # 快速验证用 50；正式训练用 300+
n_eval_episodes    = 3          # 评估 episode 数（用于电压可视化）
reward_plot_window = 20         # 奖励移动平均窗口
n_recent_to_plot   = 2          # 训练轨迹图展示的最近 episode 数

# ── 运行时 ────────────────────────────────────────────────────
seed           = 0
runtime_mode   = "performance"  # performance | strict_reproducibility
device_request = None           # None = 自动选 CUDA
require_cuda   = False

In [ ]:
cfg = compose_experiment_config(
    profile="debug",
    algorithm=algorithm,
    model_family="mlp",
    reward_type="composite",        # apply_grid_profile 会覆盖为 grid_composite
    observation_profile="simbench",
    forecast_type="perfect",
    vec_env_type="dummy",
    data_dir=project_root / "data",
    runtime_mode=runtime_mode,
    seed=seed,
    require_cuda=require_cuda,
)

# ── Grid 一键切换 ─────────────────────────────────────────────
# 设置：env_type="grid_pf", reward="grid_composite",
#       num_envs=1, vec_env_type="dummy",
#       sb_code="1-LV-rural1--0-sw", agent_bus_ids=[10,6,12]
apply_grid_profile(cfg, "rural1_phase1")

cfg.train.train_episodes = train_episodes

runtime_state = configure_torch_runtime(
    cfg, device=device_request, seed=seed, require_cuda=require_cuda
)
device_info = describe_device(runtime_state)

summary = summarize_cfg(cfg)
summary["device_info"] = device_info
summary

In [ ]:
runner = build_runner(cfg, seed=seed, env_name="GridTrain", number=1)
plot_reward_fn = runner.env_evaluate.reward_fn

print("env_type     =", cfg.env.env_type)
print("reward_type  =", cfg.reward.type)
print("num_envs     =", cfg.train.num_envs)
print("Obs schema   =", cfg.runtime.observation_schema)
print("Action dim   =", cfg.runtime.action_dim)
print("Reward metas =", [m.key for m in plot_reward_fn.component_meta])
# 期望: ['r_inc', 'r_pen', 'r_pbrs', 'r_soc', 'r_bonus', 'r_v_pen', 'r_l_pen']

In [ ]:
# ⚠ num_envs=1 + pandapower 每步 ~50ms → 比普通训练慢约 12×
# train_episodes=100, episode_limit=192 步 → 约 3~5 分钟
episodes_completed = runner.run()
print(f"训练完成：{episodes_completed} episodes")
runner.perf_summary

In [ ]:
# r_v_pen 和 r_l_pen 是 ComponentMeta 条目，自动出现在图的第 6、7 个子图
plot_reward_decomposition(
    history=runner.history,
    episode_rewards=runner.episode_rewards,
    reward_fn=plot_reward_fn,
    title="Grid Training — Reward Decomposition",
    window=reward_plot_window,
)

In [ ]:
plot_last_k_episodes_price_action_soc(
    history=runner.history,
    k=n_recent_to_plot,
    n_agents=cfg.env.num_agents,
    title_prefix="Grid Train",
)

## Grid-Specific Evaluation

使用**自定义评估循环**，同时运行两个 recorder：

- `episode_recorder`（标准）— 记录 SoC、电池功率、奖励分量
- `grid_recorder`（新增）— 记录节点电压、线路负载、违规统计

内置 `evaluate_runner` 只运行前者，无法输出电压图，因此这里手动实现循环。

In [ ]:
eval_env     = runner.env_evaluate
controller   = MADRLController(runner.agent_n, noise_std=0.0)
reward_metas = eval_env.reward_fn.component_meta

all_histories      = []
all_grid_histories = []

for ep_i in range(n_eval_episodes):
    obs_n = eval_env.reset()
    controller.reset()

    history      = init_episode_record(
        n_agents=eval_env.n,
        init_soc=float(eval_env.init_soc),
        reward_metas=reward_metas,
    )
    grid_history = init_grid_record(n_agents=eval_env.n)

    done = False
    while not done:
        action_n = controller.act(obs_n, deterministic=True)
        obs_n, r_n, done_n, info = eval_env.step(action_n)

        step_total = float(np.sum(np.asarray(r_n, dtype=np.float32)))
        append_step_record(history, info, step_total=step_total, reward_metas=reward_metas)
        append_grid_step_record(grid_history, info)

        done = bool(info.get("episode_done", False))

    all_histories.append(history)
    all_grid_histories.append(grid_history)

    n_steps = len(grid_history["pf_converged"])
    n_conv  = sum(grid_history["pf_converged"])
    n_vviol = sum(grid_history["n_v_violations"])
    n_lviol = sum(grid_history["n_l_violations"])
    print(f"Ep {ep_i+1:2d}: 步数={n_steps:3d}  潮流收敛={n_conv}/{n_steps}  "
          f"电压违规={n_vviol:3d} 步  线路违规={n_lviol:3d} 步")

total_v = sum(sum(gh["n_v_violations"]) for gh in all_grid_histories)
total_l = sum(sum(gh["n_l_violations"]) for gh in all_grid_histories)
print(f"\n汇总 — 电压违规总步数: {total_v},  线路违规总步数: {total_l}")

In [ ]:
eval_episode_rewards = [sum(h["r_total_sum"]) for h in all_histories]

plot_reward_decomposition(
    history=all_histories,
    episode_rewards=eval_episode_rewards,
    reward_fn=plot_reward_fn,
    title="Grid Evaluation — Reward Decomposition",
    window=max(1, len(all_histories)),
)

In [ ]:
# 最后一个评估 episode 的节点电压时序图
plot_node_voltages(
    grid_history=all_grid_histories[-1],
    v_min=cfg.grid.v_min_pu,
    v_max=cfg.grid.v_max_pu,
    title=f"节点电压 (pu) — Eval Ep {n_eval_episodes}",
    agent_labels=[f"Bus {bid}" for bid in cfg.grid.agent_bus_ids],
)

In [ ]:
# 复合图：顶行电压 + 每个 agent 的电池功率柱 / SoC 线
plot_battery_and_grid(
    history=all_histories[-1],
    grid_history=all_grid_histories[-1],
    n_agents=cfg.env.num_agents,
    v_min=cfg.grid.v_min_pu,
    v_max=cfg.grid.v_max_pu,
    agent_labels=[f"Bus {bid}" for bid in cfg.grid.agent_bus_ids],
    title_prefix="Grid Eval — ",
)

In [ ]:
# 所有评估 episode 的约束违反统计柱状图
plot_grid_constraint_summary(
    grid_histories=all_grid_histories,
    title=f"评估约束违反统计（共 {n_eval_episodes} 个 episode）",
)

In [ ]:
save_dir = get_madrl_checkpoint_root(project_root) / "MADDPG_Grid"
save_dir.mkdir(parents=True, exist_ok=True)
runner.save_model(str(save_dir), episode=episodes_completed)
runner.close()
print(f"模型已保存到: {save_dir}")